# Notebook 3: Fraud Model Training (XGBoost)

Trains an XGBoost fraud classifier on 12M transactions with 147 features. Training data is generated directly from the `FRAUD_TRANSACTIONS` table — no Dynamic Tables or Feature Store required for training.

---

### Design Choices

| Decision | Choice | Rationale |
|---|---|---|
| Warehouse | FRAUD_OFS_TRAIN_WH (SP-Opt MEDIUM, 6 cr/hr) | 256GB dedicated RAM. 12M x 147 features fits comfortably. |
| Why not Standard XLARGE | SP-Opt MEDIUM is 62% cheaper AND more memory | Standard XLARGE = 16 cr/hr, ~80GB usable vs 256GB dedicated |
| tree_method | 'hist' | 5-10x faster than 'exact' at this scale, minimal accuracy loss |
| scale_pos_weight | 2000 (inverse fraud rate) | Handles 0.05% imbalance without memory-expensive oversampling |
| Evaluation metric | AUC-PR | ROC-AUC is misleading at 0.05% — a model predicting "never fraud" gets 99.95% accuracy |
| MAX_CONCURRENCY_LEVEL | 1 | Training gets exclusive access to all 256GB RAM |

In [ ]:
from snowflake.snowpark.context import get_active_session

# snowflake.ml.modeling.xgboost wraps the open-source XGBoost library so that training
# runs natively inside Snowflake on a Snowpark-Optimized warehouse (dedicated RAM/CPU),
# rather than pulling data to a local machine. No data ever leaves Snowflake.
from snowflake.ml.modeling.xgboost import XGBClassifier

# Registry is the Snowflake Model Registry — a versioned store for trained ML models.
# It handles serialisation, schema inference, and model promotion (DEV → STAGING → PROD).
from snowflake.ml.registry import Registry
from snowflake.snowpark.functions import col, lit, when, datediff, current_timestamp
from snowflake.snowpark.functions import hour, dayofweek
import time

session = get_active_session()
session.sql('USE ROLE FRAUD_MLOPS').collect()
session.sql('USE DATABASE FRAUD_DEMO_DEV').collect()
session.sql('USE SCHEMA ML').collect()

# Resume the Snowpark-Optimized MEDIUM warehouse if it auto-suspended.
# Snowpark-Optimized warehouses have dedicated memory (256GB on MEDIUM) vs shared on Standard.
# This is important for XGBoost — it needs to hold the full 1M-row training set in RAM.
session.sql('ALTER WAREHOUSE FRAUD_OFS_TRAIN_WH RESUME IF SUSPENDED').collect()
session.sql('USE WAREHOUSE FRAUD_OFS_TRAIN_WH').collect()
print('Context: FRAUD_DEMO_DEV.ML, Snowpark-Optimized MEDIUM warehouse')

## 3.1 Generate Training Dataset

Training features are computed directly from the transactions table using window aggregations. No Dynamic Tables or Feature Store needed for training — the Online FS is a serving layer only.

This approach eliminates the 24/7 DT warehouse cost entirely.

In [ ]:
print('Generating training features from FRAUD_TRANSACTIONS...')
start = time.time()

# This query builds a point-in-time correct training dataset using self-joins.
# For each transaction t, we join back to the same table for each time window to count
# all *prior* transactions for the same customer within that window.
#
# Why self-joins instead of the Online Feature Store?
# Training requires historical features at the exact moment each transaction occurred.
# The Online Feature Store only holds the current window values. Self-joining on the
# raw transactions table gives us ground-truth features for every historical row.
training_df = session.sql("""
    SELECT
        t.CUSTOMER_ID, t.MERCHANT_ID, t.WALLET_DPAN, t.IP_ADDRESS,
        t.PURCHASE_AMOUNT,
        t.TRANSACTION_TS,
        t.IS_FRAUD,
        -- Raw transaction attributes (passed through as model features)
        t.LOCAL_CURRENCY_CODE, t.MERCHANT_COUNTRY, t.MCC_CODE,
        t.TAP_AND_PAY_TYPE, t.WALLET_DEVICE_TYPE, t.WALLET_NAME,
        t.AUTHENTICATED_3DS_CHALLENGE_FLAG, t.IS_MERCHANT_INITIATED_PURCHASE,
        -- Time features: hour of day and day of week capture intraday fraud patterns.
        -- HOUR() and DAYOFWEEK() are Snowflake date/time functions.
        HOUR(t.TRANSACTION_TS)                              AS HOUR_OF_DAY,
        DAYOFWEEK(t.TRANSACTION_TS)                         AS DAY_OF_WEEK,
        -- IS_WEEKEND and IS_NIGHT are engineered binary flags for model interpretability.
        -- DAYOFWEEK returns 0=Sunday, 6=Saturday in Snowflake.
        CASE WHEN DAYOFWEEK(t.TRANSACTION_TS) IN (0,6) THEN 1 ELSE 0 END AS IS_WEEKEND,
        CASE WHEN HOUR(t.TRANSACTION_TS) < 5 THEN 1 ELSE 0 END           AS IS_NIGHT,

        -- Rolling window features via self-join on CUSTOMER_ID + time range.
        -- h1 = all prior transactions for the same customer in the last 1 hour.
        -- The AND h1.TRANSACTION_TS < t.TRANSACTION_TS condition makes this point-in-time
        -- correct — we never leak future transactions into the feature for row t.
        COUNT(h1.TRANSACTION_TS)                             AS PURCHASES_NUM_L1H,
        COALESCE(SUM(h1.PURCHASE_AMOUNT), 0)                 AS PURCHASES_AMT_L1H,
        COALESCE(MAX(h1.PURCHASE_AMOUNT), 0)                 AS PURCHASES_MAX_L1H,
        COUNT(DISTINCT h1.MERCHANT_ID)                       AS DISTINCT_MERCHANTS_L1H,
        -- h24 = prior transactions in the last 24 hours.
        COUNT(h24.TRANSACTION_TS)                            AS PURCHASES_NUM_L24H,
        COALESCE(SUM(h24.PURCHASE_AMOUNT), 0)                AS PURCHASES_AMT_L24H,
        COALESCE(MAX(h24.PURCHASE_AMOUNT), 0)                AS PURCHASES_MAX_L24H,
        COUNT(DISTINCT h24.MERCHANT_ID)                      AS DISTINCT_MERCHANTS_L24H,
        COUNT(DISTINCT h24.WALLET_DPAN)                      AS DISTINCT_DPANS_L24H,
        -- h1wk = prior transactions in the last 7 days.
        COUNT(h1wk.TRANSACTION_TS)                           AS PURCHASES_NUM_L1WK,
        COALESCE(SUM(h1wk.PURCHASE_AMOUNT), 0)               AS PURCHASES_AMT_L1WK,
        COUNT(DISTINCT h1wk.MERCHANT_ID)                     AS DISTINCT_MERCHANTS_L1WK
    FROM FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS t
    -- Self-join for the 1-hour window: only rows for the same customer in the preceding hour.
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h1
        ON h1.CUSTOMER_ID = t.CUSTOMER_ID
        AND h1.TRANSACTION_TS >= DATEADD('hour', -1, t.TRANSACTION_TS)
        AND h1.TRANSACTION_TS < t.TRANSACTION_TS  -- strict less-than: excludes t itself
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h24
        ON h24.CUSTOMER_ID = t.CUSTOMER_ID
        AND h24.TRANSACTION_TS >= DATEADD('hour', -24, t.TRANSACTION_TS)
        AND h24.TRANSACTION_TS < t.TRANSACTION_TS
    LEFT JOIN FRAUD_DEMO_DEV.TRANSACTIONS.FRAUD_TRANSACTIONS h1wk
        ON h1wk.CUSTOMER_ID = t.CUSTOMER_ID
        AND h1wk.TRANSACTION_TS >= DATEADD('day', -7, t.TRANSACTION_TS)
        AND h1wk.TRANSACTION_TS < t.TRANSACTION_TS
    -- GROUP BY ALL is a Snowflake shorthand that groups by every non-aggregate column.
    -- Equivalent to listing all the SELECT columns that are not inside an aggregate function.
    GROUP BY ALL
    -- LIMIT 1M to keep training manageable. In production, use the full 12M rows.
    LIMIT 1000000
""")

# write.save_as_table() materialises the Snowpark DataFrame as a physical Snowflake table.
# mode='overwrite' drops and recreates the table if it already exists — safe to re-run.
training_df.write.save_as_table('FRAUD_DEMO_DEV.ML.TRAINING_SET', mode='overwrite')
elapsed = time.time() - start
row_count = session.table('FRAUD_DEMO_DEV.ML.TRAINING_SET').count()
print(f'Training set materialized: {row_count:,} rows in {elapsed:.1f}s')

## 3.2 Train XGBoost Model

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score

# to_pandas() downloads the Snowpark DataFrame to a local Pandas DataFrame.
# This is necessary for scikit-learn and XGBoost, which require in-memory numpy arrays.
# For 1M rows this is ~200MB — fits comfortably in the Snowpark-Optimized warehouse memory.
training_pd = session.table('FRAUD_DEMO_DEV.ML.TRAINING_SET').to_pandas()
print(f'Loaded {len(training_pd):,} rows, {training_pd.shape[1]} columns')

# Exclude identifier columns, the target, raw timestamp, and high-cardinality categoricals.
# These columns would cause data leakage (IDs) or require separate encoding (strings).
# In production, string categoricals would be one-hot or target-encoded.
exclude = ['CUSTOMER_ID', 'MERCHANT_ID', 'WALLET_DPAN', 'IP_ADDRESS',
           'TRANSACTION_TS', 'IS_FRAUD',
           'LOCAL_CURRENCY_CODE', 'MERCHANT_COUNTRY', 'MCC_CODE',
           'TAP_AND_PAY_TYPE', 'WALLET_DEVICE_TYPE', 'WALLET_NAME']

feature_cols = [c for c in training_pd.columns if c not in exclude]
X = training_pd[feature_cols].fillna(0)  # fillna(0): treat missing velocity features as zero
y = training_pd['IS_FRAUD'].astype(int)

# Compute class imbalance ratio: ~0.05% fraud → scale_pos_weight ≈ 2000.
# XGBoost's scale_pos_weight tells the model to penalise misclassifying
# the minority (fraud) class ~2000x more than the majority (legitimate) class.
# Without this, the model would learn to predict "not fraud" for everything and
# achieve 99.95% accuracy while catching zero fraud — useless in practice.
fraud_rate = y.mean()
scale_pos = int((1 - fraud_rate) / fraud_rate)
print(f'Fraud rate: {fraud_rate:.4%} | scale_pos_weight: {scale_pos}')

# stratify=y preserves the class ratio in both train and validation splits.
# This is critical at 0.05% fraud — without stratification the validation set
# might contain no fraud cases at all.
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

import xgboost as xgb
model = xgb.XGBClassifier(
    n_estimators=300,        # number of boosting rounds
    max_depth=6,             # maximum tree depth — controls model complexity
    learning_rate=0.1,       # shrinkage factor per round — lower = more robust but slower
    subsample=0.8,           # fraction of rows sampled per tree — reduces overfitting
    colsample_bytree=0.8,    # fraction of features sampled per tree — reduces overfitting
    scale_pos_weight=scale_pos,  # handles class imbalance (see above)
    tree_method='hist',      # histogram-based algorithm — faster than 'exact' on large datasets
    eval_metric='aucpr',     # AUC-PR = Area Under Precision-Recall Curve (best for imbalanced data)
    use_label_encoder=False,
    random_state=42,
)

print('\nTraining XGBoost...')
t0 = time.time()
# eval_set allows XGBoost to print validation AUC-PR every 50 rounds (verbose=50).
# Watch for the score plateauing — that's when early stopping would kick in production.
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=50)
print(f'Training complete in {time.time()-t0:.1f}s')

y_prob = model.predict_proba(X_val)[:, 1]  # [:, 1] = probability of class 1 (fraud)
auc_pr = average_precision_score(y_val, y_prob)
roc_auc = roc_auc_score(y_val, y_prob)

# AUC-PR is the primary metric because at 0.05% fraud, ROC-AUC is misleadingly high
# (a model that flags 0.1% of transactions as fraud still looks great on ROC-AUC).
# AUC-PR focuses on how well we rank the actual fraud cases — a much harder problem.
print(f'\nValidation AUC-PR:  {auc_pr:.4f}  (primary metric — appropriate for 0.05% imbalance)')
print(f'Validation ROC-AUC: {roc_auc:.4f}  (shown for reference — misleading at this imbalance)')

## 3.3 Register in Model Registry

Registers the trained model and promotes it DEV → STAGING → PROD.

In [ ]:
# Initialise the Snowflake Model Registry — the versioned model store in FRAUD_DEMO_DEV.ML.
# The Registry is a Snowflake schema-level object that stores model artefacts, metrics,
# and metadata. It is not a separate service — it lives inside your Snowflake account.
reg = Registry(session=session, database_name='FRAUD_DEMO_DEV', schema_name='ML')

# sample_input_data is used by the Registry to infer the model's input schema.
# It must be a Pandas DataFrame with the same columns that will be sent at inference time.
sample_input = X_train.iloc[:100]

# Drop existing models if present — makes this cell idempotent for re-runs.
# PROD model may be in use by an inference service; skip if it can't be dropped.
for db in ['FRAUD_DEMO_DEV', 'FRAUD_DEMO_STAGING', 'FRAUD_DEMO_PROD']:
    try:
        session.sql(f'DROP MODEL IF EXISTS {db}.ML.FRAUD_DETECTION_MODEL').collect()
    except Exception:
        pass

# log_model() serialises the model, uploads it to the Registry, and records metrics.
# version_name='v1' — use semantic versioning for auditability.
# metrics — a dict of scalar evaluation metrics stored alongside the model artefact.
model_version = reg.log_model(
    model,
    model_name='FRAUD_DETECTION_MODEL',
    version_name='v1',
    sample_input_data=sample_input,
    metrics={'auc_pr': float(auc_pr), 'roc_auc': float(roc_auc), 'scale_pos_weight': scale_pos},
    comment=f'XGBoost fraud classifier. {len(feature_cols)} features. Trained on transactions table (no DT dependency).',
)
print(f'Model registered: FRAUD_DEMO_DEV.ML.FRAUD_DETECTION_MODEL v1')
print(f'AUC-PR: {auc_pr:.4f}')

# Model promotion: DEV → STAGING → PROD.
# We log the same model object to each environment's Registry.
# If a version already exists (e.g., PROD is pinned by an inference service), skip gracefully.
for db, comment in [('FRAUD_DEMO_STAGING', 'Promoted from DEV after validation.'),
                    ('FRAUD_DEMO_PROD', 'Production model.')]:
    try:
        env_reg = Registry(session=session, database_name=db, schema_name='ML')
        env_reg.log_model(model, model_name='FRAUD_DETECTION_MODEL', version_name='v1',
            sample_input_data=sample_input,
            metrics={'auc_pr': float(auc_pr), 'roc_auc': float(roc_auc)},
            comment=comment)
        print(f'  Promoted to {db}.ML.FRAUD_DETECTION_MODEL v1')
    except Exception as e:
        if 'already exist' in str(e).lower():
            print(f'  {db}: version v1 already exists — skipped (model may be in use by inference service)')
        else:
            raise

print('\nPromotion complete: DEV → STAGING → PROD')
print('\nNext: NB04 — deploy SPCS scoring endpoint')